![Esquema relacional del modelo core](../outputs/analysis_Data.png)

# Diseño inicial de `analytics.fact_orders`

## Grano de la tabla

La tabla `analytics.fact_orders` tendrá una granularidad de **una fila por pedido**.

**1 fila = 1 `order_id`**

Esto significa que cualquier información procedente de tablas con varias filas por pedido debe agregarse previamente a nivel de `order_id`.

La estructura conceptual de la tabla será:

`fact_orders = orders + items_by_order + payments_by_order + reviews_by_order`

## Columnas propuestas para `analytics.fact_orders`

| Columna en `fact_orders` | Procedencia | Tipo | ¿Agregación? | Explicación |
|---|---|---|---|---|
| `order_id` | `core.orders.order_id` | Identificador | No | Identificador único del pedido. Será la clave principal de la tabla de hechos. |
| `customer_id` | `core.orders.customer_id` | Identificador / FK | No | Cliente asociado al pedido. Puede funcionar después como clave hacia `dim_customer`. |
| `order_status` | `core.orders.order_status` | Categórica | No | Estado del pedido. Puede convertirse después en dimensión. |
| `order_purchase_timestamp` | `core.orders.order_purchase_timestamp` | Fecha | No | Fecha y hora de compra del pedido. |
| `order_approved_at` | `core.orders.order_approved_at` | Fecha | No | Fecha y hora de aprobación del pedido. Puede contener nulos. |
| `order_delivered_carrier_date` | `core.orders.order_delivered_carrier_date` | Fecha | No | Fecha en la que el pedido fue entregado al transportista. Puede contener nulos. |
| `order_delivered_customer_date` | `core.orders.order_delivered_customer_date` | Fecha | No | Fecha real de entrega al cliente. Puede contener nulos. |
| `order_estimated_delivery_date` | `core.orders.order_estimated_delivery_date` | Fecha | No | Fecha estimada de entrega. |
| `approval_time_days` | Derivada de `core.orders` | Métrica temporal | Derivada | Tiempo entre compra y aprobación del pedido. |
| `carrier_dispatch_time_days` | Derivada de `core.orders` | Métrica temporal | Derivada | Tiempo entre aprobación y entrega al transportista. |
| `delivery_time_days` | Derivada de `core.orders` | Métrica temporal | Derivada | Tiempo entre entrega al transportista y entrega al cliente. |
| `total_delivery_time_days` | Derivada de `core.orders` | Métrica temporal | Derivada | Tiempo total desde compra hasta entrega al cliente. |
| `delay_days` | Derivada de `core.orders` | Métrica temporal | Derivada | Diferencia entre entrega real y entrega estimada. |
| `is_delayed` | Derivada de `delay_days` | Binaria | Derivada | 1 si el pedido llegó tarde, 0 si llegó a tiempo o antes. |
| `total_items_value` | `core.order_items.price` | Métrica económica | Sí | Suma del precio de todos los ítems del pedido. |
| `total_freight_value` | `core.order_items.freight_value` | Métrica económica/logística | Sí | Suma del coste de envío de todas las líneas del pedido. |
| `number_of_items` | `core.order_items.order_item_id` | Métrica de conteo | Sí | Número total de líneas de pedido asociadas al pedido. |
| `number_of_products` | `core.order_items.product_id` | Métrica de conteo | Sí | Número de productos distintos dentro del pedido. |
| `number_of_sellers` | `core.order_items.seller_id` | Métrica de conteo | Sí | Número de vendedores distintos involucrados en el pedido. |
| `total_payment_value` | `core.order_payments.payment_value` | Métrica económica | Sí | Suma del valor pagado en todas las operaciones de pago del pedido. |
| `number_of_payment_operations` | `core.order_payments.payment_sequential` | Métrica de conteo | Sí | Número de registros de pago asociados al pedido. |
| `max_payment_installments` | `core.order_payments.payment_installments` | Métrica financiera | Sí | Máximo número de cuotas usado en alguna operación de pago del pedido. |
| `avg_review_score` | `core.order_reviews.review_score` | Métrica de satisfacción | Sí | Puntuación media de las reseñas asociadas al pedido. |
| `number_of_reviews` | `core.order_reviews.review_id` | Métrica de conteo | Sí | Número de reseñas asociadas al pedido. |

# Diseño inicial de dimensiones del modelo `analytics`

La capa `analytics` utilizará dimensiones para contextualizar las tablas de hechos. Estas dimensiones permitirán analizar los pedidos y las líneas de pedido desde distintas perspectivas: cliente, vendedor, producto, estado del pedido, fecha y localización.

## Resumen de dimensiones propuestas

| Dimensión | Grano de la dimensión | Procedencia principal | ¿Agregación? | Se relaciona con | Explicación |
|---|---|---|---|---|---|
| `dim_customer` | Una fila por `customer_id` | `core.customers` | No | `fact_orders`, posiblemente `fact_order_items` | Describe al cliente asociado al pedido. Incluye identificador, cliente único, ciudad, estado y prefijo postal. |
| `dim_seller` | Una fila por `seller_id` | `core.sellers` | No | `fact_order_items` | Describe al vendedor asociado a cada línea de pedido. Se conecta mejor con `fact_order_items`, porque un pedido puede tener varios vendedores. |
| `dim_product` | Una fila por `product_id` | `core.products` | No | `fact_order_items` | Describe el producto vendido en cada línea de pedido. Incluye categoría y atributos físicos del producto. |
| `dim_order_status` | Una fila por `order_status` | `core.orders` | Sí, por valores únicos | `fact_orders` | Permite analizar los pedidos según su estado: delivered, shipped, canceled, unavailable, etc. |
| `dim_date` | Una fila por fecha | Fechas de `core.orders` y `core.order_items` | Sí, por calendario | `fact_orders`, `fact_order_items` | Permite analizar los hechos por día, mes, año, día de la semana u otras variables temporales. |
| `dim_geography` | Una fila por prefijo postal | `core.geolocations` | No | `dim_customer`, `dim_seller` o hechos si se decide conectar directamente | Permite representar la localización aproximada mediante prefijo postal, ciudad, estado, latitud y longitud. |
| `dim_payment_type` | Una fila por tipo de pago | `core.order_payments.payment_type` | Sí, por valores únicos | Opcional | Podría usarse si se decide incorporar `main_payment_type` en `fact_orders` o construir una tabla específica de pagos. En la primera versión puede quedar como dimensión opcional. |

## Columnas propuestas por dimensión

| Dimensión | Columna | Procedencia | Tipo | Explicación |
|---|---|---|---|---|
| `dim_customer` | `customer_id` | `core.customers.customer_id` | Identificador | Identificador único del cliente en el dataset. |
| `dim_customer` | `customer_unique_id` | `core.customers.customer_unique_id` | Identificador | Identificador del cliente único, útil para detectar clientes con más de un pedido. |
| `dim_customer` | `customer_zip_code_prefix` | `core.customers.customer_zip_code_prefix` | Geográfica | Prefijo postal del cliente. |
| `dim_customer` | `customer_city` | `core.customers.customer_city` | Categórica | Ciudad del cliente. |
| `dim_customer` | `customer_state` | `core.customers.customer_state` | Categórica | Estado/región del cliente. |
| `dim_seller` | `seller_id` | `core.sellers.seller_id` | Identificador | Identificador único del vendedor. |
| `dim_seller` | `seller_zip_code_prefix` | `core.sellers.seller_zip_code_prefix` | Geográfica | Prefijo postal del vendedor. |
| `dim_seller` | `seller_city` | `core.sellers.seller_city` | Categórica | Ciudad del vendedor. |
| `dim_seller` | `seller_state` | `core.sellers.seller_state` | Categórica | Estado/región del vendedor. |
| `dim_product` | `product_id` | `core.products.product_id` | Identificador | Identificador único del producto. |
| `dim_product` | `product_category_name` | `core.products.product_category_name` | Categórica | Categoría del producto. |
| `dim_product` | `product_weight_g` | `core.products.product_weight_g` | Numérica descriptiva | Peso del producto en gramos. |
| `dim_product` | `product_length_cm` | `core.products.product_length_cm` | Numérica descriptiva | Largo del producto en centímetros. |
| `dim_product` | `product_height_cm` | `core.products.product_height_cm` | Numérica descriptiva | Alto del producto en centímetros. |
| `dim_product` | `product_width_cm` | `core.products.product_width_cm` | Numérica descriptiva | Ancho del producto en centímetros. |
| `dim_product` | `product_volume_cm3` | Derivada de dimensiones físicas | Derivada | Volumen aproximado del producto: largo × alto × ancho. |
| `dim_order_status` | `order_status` | `core.orders.order_status` | Categórica | Estado del pedido. |
| `dim_date` | `date_key` | Derivada | Identificador | Clave de fecha, normalmente en formato `YYYYMMDD`. |
| `dim_date` | `full_date` | Derivada | Fecha | Fecha completa. |
| `dim_date` | `year` | Derivada | Temporal | Año. |
| `dim_date` | `month` | Derivada | Temporal | Mes numérico. |
| `dim_date` | `month_name` | Derivada | Temporal | Nombre del mes. |
| `dim_date` | `day` | Derivada | Temporal | Día del mes. |
| `dim_date` | `day_of_week` | Derivada | Temporal | Día de la semana. |
| `dim_geography` | `zip_code_prefix` | `core.geolocations.geolocation_zip_code_prefix` | Geográfica | Prefijo postal. |
| `dim_geography` | `city` | `core.geolocations.geolocation_city` | Categórica | Ciudad asociada al prefijo postal. |
| `dim_geography` | `state` | `core.geolocations.geolocation_state` | Categórica | Estado asociado al prefijo postal. |
| `dim_geography` | `latitude` | `core.geolocations.geolocation_lat` | Geográfica | Latitud aproximada del prefijo postal. |
| `dim_geography` | `longitude` | `core.geolocations.geolocation_lng` | Geográfica | Longitud aproximada del prefijo postal. |
| `dim_payment_type` | `payment_type` | `core.order_payments.payment_type` | Categórica | Tipo de pago utilizado: credit_card, boleto, voucher, debit_card, etc. |

## Relaciones principales con las tablas de hechos

| Tabla de hechos | Dimensión relacionada | Relación prevista | Comentario |
|---|---|---|---|
| `fact_orders` | `dim_customer` | `customer_id` | Cada pedido pertenece a un cliente. |
| `fact_orders` | `dim_order_status` | `order_status` | Cada pedido tiene un estado principal. |
| `fact_orders` | `dim_date` | Fechas del pedido | Puede relacionarse con varias fechas: compra, aprobación, entrega y fecha estimada. |
| `fact_order_items` | `dim_product` | `product_id` | Cada línea de pedido corresponde a un producto. |
| `fact_order_items` | `dim_seller` | `seller_id` | Cada línea de pedido está asociada a un vendedor. |
| `fact_order_items` | `dim_customer` | `customer_id` | Puede heredarse desde `orders` para analizar líneas por cliente. |
| `dim_customer` | `dim_geography` | `customer_zip_code_prefix` | Permite enriquecer el cliente con información territorial. |
| `dim_seller` | `dim_geography` | `seller_zip_code_prefix` | Permite enriquecer el vendedor con información territorial. |

## Nota de diseño

En esta primera versión, `dim_seller` y `dim_product` se conectarán principalmente con `fact_order_items`, no directamente con `fact_orders`, porque un mismo pedido puede contener varios productos y varios vendedores.

La tabla `fact_orders` conservará medidas agregadas como `number_of_products`, `number_of_sellers`, `total_items_value` y `total_freight_value`, mientras que el análisis detallado por producto y vendedor se realizará desde `fact_order_items`.

`dim_payment_type` queda como dimensión opcional. Para conectarla limpiamente con `fact_orders`, sería necesario definir una regla como `main_payment_type`, ya que un mismo pedido puede tener más de un tipo de pago.

Los textos de reseñas, como `review_comment_title` y `review_comment_message`, no se incluyen inicialmente como dimensiones. Se conservan en la capa `core` y podrían utilizarse en un análisis posterior de texto o satisfacción del cliente.

## Dimensiones seleccionadas para la primera versión

Para la primera versión de la constelación analítica ligera se priorizan las siguientes dimensiones:

1. `dim_customer`
2. `dim_seller`
3. `dim_product`
4. `dim_order_status`
5. `dim_date`
6. `dim_geography`

La dimensión `dim_payment_type` se mantiene como opcional para una iteración posterior.

## Modelo de hechos: `analytics.fact_order_items`

La tabla `analytics.fact_order_items` representa el análisis a nivel de línea de pedido. Su objetivo es permitir estudiar el comportamiento de los productos, vendedores, precios y costes de envío dentro de cada pedido.

A diferencia de `analytics.fact_orders`, que trabaja a nivel de pedido completo, esta tabla permite analizar el detalle interno de cada pedido, especialmente cuando un mismo pedido contiene varios productos, varios vendedores o varias líneas de compra.

### Grano de la tabla

El grano de `analytics.fact_order_items` es:

    1 fila = 1 línea de pedido = 1 combinación de order_id + order_item_id

Esto significa que cada fila representa un producto vendido dentro de un pedido concreto.

La clave primaria lógica de esta tabla es:

    order_id + order_item_id

### Procedencia de los datos

La tabla se construye principalmente a partir de:

    core.order_items

y se conecta con otras entidades mediante claves hacia las dimensiones analíticas.

Conceptualmente:

    core.order_items
    ↓
    analytics.fact_order_items

### Columnas propuestas

| Columna | Procedencia | Tipo | Descripción |
|---|---|---|---|
| `order_id` | `core.order_items.order_id` | FK | Identificador del pedido al que pertenece la línea |
| `order_item_id` | `core.order_items.order_item_id` | Identificador | Número de línea dentro del pedido |
| `customer_id` | `core.orders.customer_id` | FK | Cliente asociado al pedido |
| `product_id` | `core.order_items.product_id` | FK | Producto vendido en la línea de pedido |
| `seller_id` | `core.order_items.seller_id` | FK | Vendedor responsable de esa línea |
| `shipping_limit_date` | `core.order_items.shipping_limit_date` | Fecha | Fecha límite de envío de la línea |
| `shipping_date_key` | Derivada de `shipping_limit_date` | FK | Clave de fecha para conectar con `dim_date` |
| `price` | `core.order_items.price` | Métrica económica | Precio del producto en esa línea |
| `freight_value` | `core.order_items.freight_value` | Métrica logística/económica | Coste de envío asociado a esa línea |
| `line_total_value` | Derivada de `price + freight_value` | Métrica económica | Valor total de la línea, sumando producto y envío |

### Métricas principales

Las principales métricas de `fact_order_items` son:

| Métrica | Descripción |
|---|---|
| `price` | Valor económico del producto vendido en la línea |
| `freight_value` | Coste de envío asociado a la línea |
| `line_total_value` | Valor total de la línea, sumando producto y envío |

Estas métricas permiten analizar, por ejemplo:

- Qué productos generan más ingresos.
- Qué vendedores concentran más valor.
- Qué categorías tienen mayores costes de envío.
- Qué relación existe entre precio del producto y coste logístico.
- Qué peso tienen los costes de envío sobre el valor total de la línea.

### Dimensiones relacionadas

La tabla `analytics.fact_order_items` se relaciona con las siguientes dimensiones:

    analytics.dim_customer
    analytics.dim_product
    analytics.dim_seller
    analytics.dim_date

También se relaciona con la tabla `analytics.fact_orders`, ya que cada línea de pedido pertenece a un pedido completo.

---

## Dimensiones asociadas a `fact_order_items`

### `analytics.dim_customer`

La dimensión `dim_customer` describe al cliente asociado al pedido.

Aunque la línea de pedido procede de `core.order_items`, el `customer_id` se obtiene a través de `core.orders`, porque el cliente está asociado al pedido completo, no directamente a la línea de pedido.

#### Grano

    1 fila = 1 customer_id

#### Columnas principales

| Columna | Descripción |
|---|---|
| `customer_id` | Identificador del cliente en el pedido |
| `customer_unique_id` | Identificador único del cliente |
| `customer_zip_code_prefix` | Prefijo postal del cliente |
| `customer_city` | Ciudad del cliente |
| `customer_state` | Estado del cliente |

#### Relación con `fact_order_items`

    dim_customer 1:N fact_order_items

Clave:

    customer_id

Un cliente puede aparecer en muchas líneas de pedido, pero cada línea de pedido pertenece a un único cliente.

---

### `analytics.dim_product`

La dimensión `dim_product` describe los productos vendidos.

Esta dimensión es fundamental para analizar el rendimiento por producto, categoría, peso, volumen o características físicas.

#### Grano

    1 fila = 1 product_id

#### Columnas principales

| Columna | Descripción |
|---|---|
| `product_id` | Identificador del producto |
| `product_category_name` | Categoría original del producto |
| `product_category_name_english` | Categoría traducida al inglés, si se incorpora la tabla de traducción |
| `product_weight_g` | Peso del producto en gramos |
| `product_length_cm` | Longitud del producto en centímetros |
| `product_height_cm` | Altura del producto en centímetros |
| `product_width_cm` | Anchura del producto en centímetros |
| `product_volume_cm3` | Volumen calculado del producto |

#### Relación con `fact_order_items`

    dim_product 1:N fact_order_items

Clave:

    product_id

Un producto puede aparecer en muchas líneas de pedido, pero cada línea de pedido hace referencia a un único producto.

#### Nota conceptual

`dim_product` no debe conectarse directamente con `fact_orders`, porque un pedido puede contener varios productos. La relación correcta entre pedidos y productos se resuelve mediante `fact_order_items`.

---

### `analytics.dim_seller`

La dimensión `dim_seller` describe al vendedor responsable de cada línea de pedido.

Es especialmente importante porque un mismo pedido puede contener productos de diferentes vendedores. Por eso, el análisis por vendedor debe hacerse desde `fact_order_items`, no desde `fact_orders`.

#### Grano

    1 fila = 1 seller_id

#### Columnas principales

| Columna | Descripción |
|---|---|
| `seller_id` | Identificador del vendedor |
| `seller_zip_code_prefix` | Prefijo postal del vendedor |
| `seller_city` | Ciudad del vendedor |
| `seller_state` | Estado del vendedor |

#### Relación con `fact_order_items`

    dim_seller 1:N fact_order_items

Clave:

    seller_id

Un vendedor puede aparecer en muchas líneas de pedido, pero cada línea de pedido tiene un único vendedor asociado.

#### Nota conceptual

`dim_seller` no debe conectarse directamente con `fact_orders`, porque un pedido puede tener varios vendedores. La relación correcta es:

    fact_orders 1:N fact_order_items N:1 dim_seller

---

### `analytics.dim_date`

La dimensión `dim_date` permite analizar las líneas de pedido desde una perspectiva temporal.

En el caso de `fact_order_items`, la fecha principal es:

    shipping_limit_date

A partir de esta fecha se genera una clave:

    shipping_date_key

#### Grano

    1 fila = 1 fecha

#### Columnas principales

| Columna | Descripción |
|---|---|
| `date_key` | Clave de fecha, normalmente en formato `YYYYMMDD` |
| `full_date` | Fecha completa |
| `year` | Año |
| `month` | Mes |
| `month_name` | Nombre del mes |
| `day` | Día del mes |
| `day_of_week` | Día de la semana |
| `quarter` | Trimestre |

#### Relación con `fact_order_items`

    dim_date 1:N fact_order_items

Clave:

    date_key → shipping_date_key

Una fecha puede estar asociada a muchas líneas de pedido, pero cada línea de pedido tiene una única fecha límite de envío.

---

## Relación con `analytics.fact_orders`

La tabla `fact_order_items` también se relaciona con `fact_orders`.

### Relación

    fact_orders 1:N fact_order_items

Clave:

    order_id

Esto significa que un pedido puede contener una o varias líneas de pedido.

Esta relación permite analizar el detalle de productos y vendedores sin perder la conexión con el pedido completo.

Por ejemplo:

    fact_orders
      └── fact_order_items
            ├── dim_product
            ├── dim_seller
            ├── dim_customer
            └── dim_date

---

## Relaciones correctas de `fact_order_items`

| Origen | Destino | Tipo | Clave |
|---|---|---|---|
| `analytics.fact_orders` | `analytics.fact_order_items` | 1:N | `order_id` |
| `analytics.dim_customer` | `analytics.fact_order_items` | 1:N | `customer_id` |
| `analytics.dim_product` | `analytics.fact_order_items` | 1:N | `product_id` |
| `analytics.dim_seller` | `analytics.fact_order_items` | 1:N | `seller_id` |
| `analytics.dim_date` | `analytics.fact_order_items` | 1:N | `date_key → shipping_date_key` |

---

## Relaciones que no deben aparecer

En el modelo analítico no deben aparecer las siguientes relaciones directas:

| Relación incorrecta | Motivo |
|---|---|
| `dim_product → fact_orders` | Un pedido puede tener varios productos |
| `dim_seller → fact_orders` | Un pedido puede tener varios vendedores |
| `dim_geography → fact_order_items` | La geografía se conecta a través de cliente o vendedor |
| `dim_order_status → fact_order_items` | El estado pertenece al pedido completo, no a la línea |
| `dim_date → dim_customer` | La fecha no describe directamente al cliente |

---

## Justificación del diseño

La tabla `fact_order_items` se incluye porque el análisis logístico del marketplace no puede limitarse únicamente al nivel de pedido. Muchos pedidos contienen varios productos y pueden involucrar a varios vendedores, por lo que una única tabla de hechos a nivel pedido ocultaría información relevante.

La existencia de `fact_order_items` permite conservar una granularidad más detallada para analizar precios, costes de envío, productos, categorías y vendedores. Al mismo tiempo, la tabla `fact_orders` mantiene una visión agregada del pedido completo.

Por tanto, el modelo adopta una estructura de constelación ligera con dos tablas de hechos principales:

    fact_orders
    fact_order_items

Cada una responde a una granularidad diferente:

| Tabla de hechos | Grano | Uso principal |
|---|---|---|
| `fact_orders` | 1 fila por pedido | Análisis logístico global del pedido |
| `fact_order_items` | 1 fila por línea de pedido | Análisis de productos, vendedores, precios y costes de envío |

Este diseño evita duplicaciones, mantiene la trazabilidad del modelo y permite responder preguntas analíticas tanto a nivel agregado como a nivel detallado.